# SEA-AD dataset tutorial -- Integrating multiple ST slides in your analysis

After you have done the maxfuse imputation (or any other ones), train the GITIII deep learning model (see the reproducible codes for the SEA-AD dataset, you can jointly analyze multiple ST slides by:

## import necessary packages

In [ ]:
receiver_type='Astrocyte'

import warnings
warnings.simplefilter("ignore")

# Suppose that your data path is:
df_path="./raw_data/AD.csv"

import torch
import numpy as np
import random
import os
import scanpy as sc
import pandas as pd

def set_random_seed(seed: int):
    # Set seed for Python's built-in random module
    random.seed(seed)

    # Set seed for NumPy
    np.random.seed(seed)

    # Set seed for PyTorch (both CPU and GPU)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)  # if using multi-GPU.

    # Ensure deterministic behavior in PyTorch
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

    # Optionally set the seed for Scanpy (if any stochastic behavior occurs)
    sc.settings.n_jobs = 1  # To avoid parallel execution randomness

    # Set the environment variable for deterministic behavior
    os.environ['PYTHONHASHSEED'] = str(seed)

    print(f"Random seed set to {seed} for reproducibility.")

set_random_seed(42)

# Import necessary python packages
import gitiii_ag

# multiple ST slides to analysis
sections=[
    "H20.33.001.CX28.MTG.02.007.1.02.03",
    "H21.33.001.Cx22.MTG.02.007.1.01.03",
    "H21.33.031.CX24.MTG.02.007.1.01.01",
    "H20.33.040.Cx25.MTG.02.007.1.01.04",
    "H21.33.015.Cx26.MTG.02.007.1.1",
    "H21.33.040.Cx22.MTG.02.007.3.03.04",
    "H21.33.023.Cx26.MTG.02.007.1.03.04",
    "H20.33.001.Cx28.MTG.02.007.1.01.03"
]

_adata_cache={}
_match_df_cache={}

# Init the pathway analyzers for each ST section

In [ ]:
def build_pathway_analyzer_multi_sample(sections, genes_to_remove=None):
    analyzers=[]
    for section in sections:
        patient='.'.join(section.split('.')[:3])
        if patient not in _adata_cache:
            _adata_cache[patient]=sc.read_h5ad(f"./raw_data/{patient}.h5ad")
        sc_adata=_adata_cache[patient].copy()
        kwargs=dict(
            sc_adata=sc_adata,
            st_sample=section,
            species="human",
            sc_label=None,
            st_label=None,
            filter_noise_proportion=0.02,
            discard_no_match_threshold=0.7,
            num_neighbors=50,
        )
        if genes_to_remove is not None:
            kwargs["genes_to_remove_from_LR"]=genes_to_remove
        pathway_analyzer=gitiii_ag.pathway_analyzer.Pathway_analyzer(**kwargs)
        if section not in _match_df_cache:
            _match_df_cache[section]=pd.read_csv(f"./data/{section}_match.csv")
        pathway_analyzer.integrate_sc_st_with_match_df(match_df=_match_df_cache[section].copy(),check=False)
        analyzers.append(pathway_analyzer)
    return gitiii_ag.pathway_analyzer_multi_samples.Pathway_analyzer_multi_samples(analyzers)

pathway_analyzer_multi_sample=build_pathway_analyzer_multi_sample(sections)

## Build a joint pathway analyzer

In [ ]:
targeted_gene='CD74'

# Build a pathway analyzer with the receptor removed from LR pathways for this gene
current_analyzer=build_pathway_analyzer_multi_sample(sections, genes_to_remove=[targeted_gene])

## Infer the upstream LR pathway similar to the one-section part

In [ ]:
significant_LR1,significant_LR1_to_zero_order=pathway_analyzer.find_significant_LR_with_known_receiver_value__scaled_LR_VS_cell_state(receiver_type=receiver_type,targeted_gene=targeted_gene,visualize=True)